In [1]:
! pip install requests beautifulsoup4 selenium

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException
from bs4 import BeautifulSoup
import pandas as pd
import uuid
import time
import random
import re

# Настройка Chrome с более реалистичными параметрами
options = Options()
# options.add_argument("--headless")  # Раскомментируйте для запуска в фоновом режиме после отладки
options.add_argument("--disable-blink-features=AutomationControlled")  # Скрываем автоматизацию
options.add_argument("--window-size=1920,1080")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/112.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

product_urls = {
    "axis-y сыворотка": "https://kaspi.kz/shop/p/axis-y-syvorotka-dark-spot-correcting-glow-dlja-litsa-50-ml-104067981/?c=750000000",
    "велик мечты": "https://kaspi.kz/shop/p/gestalt-hx1000-9027-27-5-djuim-2022-seryi-105318549/?c=750000000",
    "ватный диск": "https://kaspi.kz/shop/p/bella-vatnye-diski-cotton-120-sht-100224746/?c=750000000",
    "омега3": "https://kaspi.kz/shop/p/now-omega-3-1000-mg-omega-3-kapsuly-100-sht-107545548/?c=750000000",
    "айфончик": "https://kaspi.kz/shop/p/apple-iphone-13-128gb-chernyi-102298404/?c=750000000"
}

def human_like_scroll(driver):
    scroll_height = driver.execute_script("return document.body.scrollHeight")
    current_position = driver.execute_script("return window.pageYOffset")
    
    while current_position < scroll_height - 500:
        step = random.randint(300, 700)
        driver.execute_script(f"window.scrollBy(0, {step});")
        time.sleep(random.uniform(0.2, 0.5))
        current_position = driver.execute_script("return window.pageYOffset")
        
        if random.random() > 0.7:
            scroll_height = driver.execute_script("return document.body.scrollHeight")

def check_reviews_tab(driver):
    try:
        tabs = WebDriverWait(driver, 5).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".tab-content-title"))
        )
        
        for tab in tabs:
            if "Отзывы" in tab.text:
                return tab
        return None
    except:
        return None

def click_show_more(driver):
    human_like_scroll(driver)
    time.sleep(1)
    
    js_check_button = """
    function findShowMoreButton() {
        // Ищем кнопки по тексту содержимого
        const allButtons = Array.from(document.querySelectorAll('button, a.button, div[role="button"]'));
        
        for (const button of allButtons) {
            const text = button.textContent.toLowerCase().trim();
            if (text.includes('показать еще') || 
                text.includes('показать ещё') || 
                text.includes('показать больше') || 
                text.includes('загрузить еще')) {
                return button;
            }
        }
        
        // Ищем по className или id, которые могут указывать на эту функциональность
        const potentialElements = Array.from(document.querySelectorAll('[class*="more"], [class*="load"], [id*="more"], [id*="load"]'));
        for (const el of potentialElements) {
            if (el.textContent.toLowerCase().includes('показ') || 
                el.textContent.toLowerCase().includes('еще') || 
                el.textContent.toLowerCase().includes('ещё')) {
                return el;
            }
        }
        
        return null;
    }
    
    const showMoreButton = findShowMoreButton();
    if (showMoreButton) {
        showMoreButton.scrollIntoView({block: 'center', behavior: 'smooth'});
        return true;
    }
    return false;
    """
    
    button_found = driver.execute_script(js_check_button)
    
    if button_found:
        try:
            show_more_xpaths = [
                "//button[contains(., 'Показать ещё')]",
                "//button[contains(., 'Показать еще')]",
                "//div[contains(@class, 'show-more')]//button",
                "//div[contains(@class, 'load-more')]//button",
                "//a[contains(., 'Показать ещё')]",
                "//span[contains(., 'Показать ещё')]/parent::*"
            ]
            
            for xpath in show_more_xpaths:
                try:
                    button = WebDriverWait(driver, 3).until(
                        EC.element_to_be_clickable((By.XPATH, xpath))
                    )
                    print(f"Найдена кнопка с XPath: {xpath}")
                    driver.execute_script("arguments[0].click();", button)
                    time.sleep(random.uniform(3, 5))
                    return True
                except:
                    continue
            
            js_click = """
            const showMoreButton = findShowMoreButton();
            if (showMoreButton) {
                showMoreButton.click();
                return true;
            }
            return false;
            """
            
            clicked = driver.execute_script(js_click)
            if clicked:
                print("Успешно кликнули на кнопку через JavaScript")
                time.sleep(random.uniform(3, 5))
                return True
                
            css_selectors = [
                "button:contains('Показать')", 
                ".show-more", 
                ".load-more", 
                "[class*='more']", 
                "[class*='show']"
            ]
            
            for selector in css_selectors:
                try:
                    js_find_by_css = f"""
                    const elements = document.querySelectorAll("{selector}");
                    for (const el of elements) {{
                        if (el.textContent.includes('Показать')) {{
                            el.click();
                            return true;
                        }}
                    }}
                    return false;
                    """
                    clicked = driver.execute_script(js_find_by_css)
                    if clicked:
                        print(f"Успешно кликнули на кнопку через CSS: {selector}")
                        time.sleep(random.uniform(3, 5))
                        return True
                except:
                    continue
                    
        except Exception as e:
            print(f"Ошибка при попытке клика на кнопку: {e}")
    
    print("Пробуем эмулировать прокрутку для активации бесконечной загрузки...")
    try:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        
        for _ in range(3):
            driver.execute_script("window.scrollBy(0, 100);")
            time.sleep(1)
        
        previous_html = driver.page_source
        time.sleep(3)
        current_html = driver.page_source
        
        if previous_html != current_html:
            print("Похоже, сработала бесконечная прокрутка")
            return True
    except Exception as e:
        print(f"Ошибка при эмуляции прокрутки: {e}")
    
    try:
        pagination = driver.find_elements(By.CSS_SELECTOR, "[class*='pagination'] a, [class*='pages'] a, .page-numbers")
        if pagination:
            for page_btn in pagination:
                if page_btn.text.isdigit() and int(page_btn.text) > 1:
                    driver.execute_script("arguments[0].click();", page_btn)
                    print(f"Переход на страницу {page_btn.text}")
                    time.sleep(3)
                    return True
    except Exception as e:
        print(f"Ошибка при поиске пагинации: {e}")
    
    try:
        iframes = driver.find_elements(By.TAG_NAME, "iframe")
        if iframes:
            for iframe in iframes:
                try:
                    driver.switch_to.frame(iframe)
                    button = driver.find_element(By.XPATH, "//button[contains(., 'Показать')]")
                    driver.execute_script("arguments[0].click();", button)
                    driver.switch_to.default_content()
                    time.sleep(3)
                    return True
                except:
                    driver.switch_to.default_content()
                    continue
    except:
        driver.switch_to.default_content()
    
    return False

def is_end_of_reviews(driver, current_reviews_count, attempts=3):
    for _ in range(attempts):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        
        soup = BeautifulSoup(driver.page_source, "html.parser")
        reviews = soup.find_all("div", class_="reviews__review")
        
        if len(reviews) > current_reviews_count:
            return False  
    
    page_text = driver.page_source.lower()
    end_phrases = ["больше отзывов нет", "показаны все отзывы", "конец списка"]
    
    for phrase in end_phrases:
        if phrase in page_text:
            print(f"Обнаружен индикатор конца списка: '{phrase}'")
            return True
    
    return True  

all_reviews = []

for product_name, url in product_urls.items():
    print(f"\nОбрабатываем {product_name}: {url}")
    driver.get(url)
    time.sleep(random.uniform(5, 8))  
    
    reviews_tab = check_reviews_tab(driver)
    if reviews_tab:
        print("Найдена вкладка с отзывами, переходим к ней")
        driver.execute_script("arguments[0].click();", reviews_tab)
        time.sleep(3)
    else:
        try:
            review_links = driver.find_elements(By.XPATH, "//a[contains(@href, 'review') or contains(@href, 'отзыв') or contains(text(), 'Отзыв')]")
            if review_links:
                for link in review_links:
                    if "отзыв" in link.text.lower() or "review" in link.text.lower():
                        driver.execute_script("arguments[0].click();", link)
                        print("Перешли по ссылке на отзывы")
                        time.sleep(3)
                        break
            else:
                print("Не удалось найти ссылку на отзывы, пробуем продолжить с текущей страницы")
        except Exception as e:
            print(f"Ошибка при поиске ссылки на отзывы: {e}")
            print("Продолжаем с текущей страницы")
    
    human_like_scroll(driver)
    time.sleep(2)
    
    collected = 0
    processed_reviews = set() 
    retry_count = 0
    
    while collected < 50 and retry_count < 5:  
        soup = BeautifulSoup(driver.page_source, "html.parser")
        reviews = soup.find_all("div", class_="reviews__review")
        
        if not reviews:
            print(f"Нет отзывов для {product_name}")
            break
        
        print(f"Найдено {len(reviews)} отзывов на текущей странице для {product_name}")
        
        new_reviews_found = False
        for review in reviews:
            review_content = review.get_text(strip=True)
            review_hash = hash(review_content)
            
            if review_hash not in processed_reviews and review_content.strip():
                processed_reviews.add(review_hash)
                new_reviews_found = True
                
                review_id = str(uuid.uuid4())
                
                date_tag = review.find("div", class_="reviews__date")
                date = date_tag.text.strip() if date_tag and date_tag.text else None
                
                comment_tag = review.find("div", class_="reviews__review-text")
                comment = None
                if comment_tag:
                    comment_text = comment_tag.get_text(strip=True)
                    comment = re.sub(r'^Комментарий:\s*', '', comment_text).strip()
                
                rating_tag = review.find("div", class_=lambda c: c and isinstance(c, str) and "rating" in c)
                rating = None
                if rating_tag:
                    classes = rating_tag.get("class", [])
                    for cls in classes:
                        if isinstance(cls, str) and cls.startswith("_") and cls[1:].isdigit():
                            try:
                                rating = int(cls[1:]) / 10
                            except ValueError:
                                rating = None
                
                all_reviews.append({
                    "ID": review_id,
                    "Атауы": product_name,
                    "Күні": date,
                    "Баға": rating,
                    "Пікір": comment
                })
                
                collected += 1
                if collected >= 50:
                    break
        
        if collected >= 50:
            print(f"Собрали 50 отзывов для {product_name}, переходим к следующему продукту")
            break
        
        if not new_reviews_found:
            retry_count += 1
            print(f"Не найдено новых отзывов. Попытка {retry_count} из 5")
            
            if retry_count >= 5 or is_end_of_reviews(driver, len(processed_reviews)):
                print("Достигнут конец списка отзывов или превышено количество попыток")
                break
        else:
            retry_count = 0
        
        if not click_show_more(driver):
            retry_count += 1
            print(f"Не удалось загрузить больше отзывов. Попытка {retry_count} из 5")
            time.sleep(2)
            
            if retry_count >= 5:
                print("Превышено количество попыток загрузки дополнительных отзывов")
                break
    
    print(f"Собрано {collected} отзывов для {product_name}")

driver.quit()

df = pd.DataFrame(all_reviews)
df.to_csv("Kaspi_pikirleri.csv", index=False, encoding='utf-8-sig')
print(f"{len(all_reviews)} пікір сәтті сақталды!")


Обрабатываем axis-y сыворотка: https://kaspi.kz/shop/p/axis-y-syvorotka-dark-spot-correcting-glow-dlja-litsa-50-ml-104067981/?c=750000000
Найдено 27 отзывов на текущей странице для axis-y сыворотка
Найдена кнопка с XPath: //a[contains(., 'Показать ещё')]
Найдено 99 отзывов на текущей странице для axis-y сыворотка
Собрали 50 отзывов для axis-y сыворотка, переходим к следующему продукту
Собрано 50 отзывов для axis-y сыворотка

Обрабатываем велик мечты: https://kaspi.kz/shop/p/gestalt-hx1000-9027-27-5-djuim-2022-seryi-105318549/?c=750000000
Найдено 45 отзывов на текущей странице для велик мечты
Найдена кнопка с XPath: //a[contains(., 'Показать ещё')]
Найдено 153 отзывов на текущей странице для велик мечты
Собрали 50 отзывов для велик мечты, переходим к следующему продукту
Собрано 50 отзывов для велик мечты

Обрабатываем ватный диск: https://kaspi.kz/shop/p/bella-vatnye-diski-cotton-120-sht-100224746/?c=750000000
Найдено 99 отзывов на текущей странице для ватный диск
Собрали 50 отзывов дл

In [3]:
df2 = pd.read_csv('Kaspi_pikirleri.csv')
df2

,ID,Атауы,Күні,Баға,Пікір
0,2fcc4eda-5ffb-4ba5-a383-8340c27536b6,axis-y сыворотка,16.10.2022,5.0,"Достоинства:Найс, обычно не оставляю отзыв (ле..."
1,012e869d-8344-47f8-9eb6-cf77556681ef,axis-y сыворотка,16.08.2023,5.0,Мен қорқып едім подделкасын салып жіберетін шы...
2,230c01de-04cd-40bf-a76b-0723650f501a,axis-y сыворотка,28.06.2024,1.0,"Отзывтарға қарап заказ бергем, подделкасын сал..."
3,548759eb-9644-40ee-ba05-4bf6c05f954e,axis-y сыворотка,27.06.2023,5.0,Достоинства:Все супер! Доставили быстро и поло...
4,f35665e2-bb77-4f10-82f8-975dc1d94add,axis-y сыворотка,29.03.2023,5.0,Спасибо магазину Красотка и Каспи за предостав...
...,...,...,...,...,...
245,d2f06661-671f-4418-93ab-de5012d613ff,скейт мечты,06.04.2024,5.0,"Скейт хороший, мягкое катание, устойчивый, пов..."
246,4518d9ac-92e7-478c-b34d-e5170811cfec,скейт мечты,04.04.2024,5.0,"Клёвый скейт, пришёл вовремя.0 человек(а) посч..."
247,b5cce813-7054-4cb7-bddc-9b40e959c48e,скейт мечты,29.03.2024,5.0,Спасибо! Сыну понравилась! Доставка быстро!0 ч...
248,b46bc654-54bd-42d8-9bdf-ed14cc993806,скейт мечты,01.01.2024,5.0,"Классный, большой.0 человек(а) посчитал(и) отз..."


In [7]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import uuid
import time
import random
import re

# Настройка Chrome с реалистичными параметрами
options = Options()
# options.add_argument("--headless")  # Раскомментируйте для запуска в фоновом режиме после отладки
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_argument("--window-size=1920,1080")
options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/112.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

product_urls = {
    "axis-y сыворотка": "https://kaspi.kz/shop/p/axis-y-syvorotka-dark-spot-correcting-glow-dlja-litsa-50-ml-104067981/?c=750000000",
    "велик мечты": "https://kaspi.kz/shop/p/gestalt-hx1000-9027-27-5-djuim-2022-seryi-105318549/?c=750000000",
    "ватный диск": "https://kaspi.kz/shop/p/bella-vatnye-diski-cotton-120-sht-100224746/?c=750000000",
    "омега3": "https://kaspi.kz/shop/p/now-omega-3-1000-mg-omega-3-kapsuly-100-sht-107545548/?c=750000000",
    "айфончик": "https://kaspi.kz/shop/p/apple-iphone-13-128gb-chernyi-102298404/?c=750000000"
}

def human_like_scroll(driver):
    scroll_height = driver.execute_script("return document.body.scrollHeight")
    current_position = driver.execute_script("return window.pageYOffset")
    
    while current_position < scroll_height - 500:
        step = random.randint(300, 700)
        driver.execute_script(f"window.scrollBy(0, {step});")
        time.sleep(random.uniform(0.2, 0.5))
        current_position = driver.execute_script("return window.pageYOffset")
        
        if random.random() > 0.7:
            scroll_height = driver.execute_script("return document.body.scrollHeight")

def click_show_more(driver):
    human_like_scroll(driver)
    time.sleep(1)
    
    try:
        show_more_xpaths = [
            "//button[contains(., 'Показать ещё')]",
            "//button[contains(., 'Показать еще')]",
            "//div[contains(@class, 'show-more')]//button",
            "//div[contains(@class, 'load-more')]//button",
            "//a[contains(., 'Показать ещё')]",
            "//span[contains(., 'Показать ещё')]/parent::*"
        ]
        
        for xpath in show_more_xpaths:
            try:
                button = WebDriverWait(driver, 3).until(
                    EC.element_to_be_clickable((By.XPATH, xpath))
                )
                print(f"Найдена кнопка с XPath: {xpath}")
                driver.execute_script("arguments[0].click();", button)
                time.sleep(random.uniform(3, 5))
                return True
            except:
                continue
                
    except Exception as e:
        print(f"Ошибка при попытке клика на кнопку: {e}")
    
    # Пробуем прокрутку для бесконечной загрузки
    try:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)
        
        for _ in range(3):
            driver.execute_script("window.scrollBy(0, 100);")
            time.sleep(1)
    except Exception as e:
        print(f"Ошибка при эмуляции прокрутки: {e}")
    
    return False

all_reviews = []

for product_name, url in product_urls.items():
    print(f"\nОбрабатываем {product_name}: {url}")
    driver.get(url)
    time.sleep(random.uniform(5, 8))
    
    # Проверяем наличие отзывов на странице или ищем ссылку на отзывы
    try:
        review_links = driver.find_elements(By.XPATH, "//a[contains(@href, 'review') or contains(@href, 'отзыв') or contains(text(), 'Отзыв')]")
        if review_links:
            for link in review_links:
                if "отзыв" in link.text.lower() or "review" in link.text.lower():
                    driver.execute_script("arguments[0].click();", link)
                    print("Перешли по ссылке на отзывы")
                    time.sleep(3)
                    break
        else:
            print("Не удалось найти ссылку на отзывы, пробуем продолжить с текущей страницы")
    except Exception as e:
        print(f"Ошибка при поиске ссылки на отзывы: {e}")
        print("Продолжаем с текущей страницы")
    
    human_like_scroll(driver)
    time.sleep(2)
    
    collected = 0
    processed_reviews = set()
    retry_count = 0
    
    while collected < 100 and retry_count < 5:
        soup = BeautifulSoup(driver.page_source, "html.parser")
        reviews = soup.find_all("div", class_="reviews__review")
        
        if not reviews:
            print(f"Нет отзывов для {product_name}")
            break
        
        print(f"Найдено {len(reviews)} отзывов на текущей странице для {product_name}")
        
        new_reviews_found = False
        for review in reviews:
            review_content = review.get_text(strip=True)
            review_hash = hash(review_content)
            
            if review_hash not in processed_reviews and review_content.strip():
                processed_reviews.add(review_hash)
                new_reviews_found = True
                
                review_id = str(uuid.uuid4())
                
                date_tag = review.find("div", class_="reviews__date")
                date = date_tag.text.strip() if date_tag and date_tag.text else None
                
                comment_tag = review.find("div", class_="reviews__review-text")
                comment = None
                if comment_tag:
                    comment_text = comment_tag.get_text(strip=True)
                    comment = re.sub(r'^Комментарий:\s*', '', comment_text).strip()
                
                rating_tag = review.find("div", class_=lambda c: c and isinstance(c, str) and "rating" in c)
                rating = None
                if rating_tag:
                    classes = rating_tag.get("class", [])
                    for cls in classes:
                        if isinstance(cls, str) and cls.startswith("_") and cls[1:].isdigit():
                            try:
                                rating = int(cls[1:]) / 10
                            except ValueError:
                                rating = None
                
                all_reviews.append({
                    "ID": review_id,
                    "Атауы": product_name,
                    "Күні": date,
                    "Баға": rating,
                    "Пікір": comment
                })
                
                collected += 1
                if collected >= 100:
                    break
        
        if collected >= 100:
            print(f"Собрали 100 отзывов для {product_name}, переходим к следующему продукту")
            break
        
        if not new_reviews_found:
            retry_count += 1
            print(f"Не найдено новых отзывов. Попытка {retry_count} из 5")
            
            if retry_count >= 5:
                print("Превышено количество попыток")
                break
        else:
            retry_count = 0
        
        if not click_show_more(driver):
            retry_count += 1
            print(f"Не удалось загрузить больше отзывов. Попытка {retry_count} из 5")
            time.sleep(2)
            
            if retry_count >= 5:
                print("Превышено количество попыток загрузки дополнительных отзывов")
                break
    
    print(f"Собрано {collected} отзывов для {product_name}")

driver.quit()

df = pd.DataFrame(all_reviews)
df.to_csv("kaspi_reviews.csv", index=False, encoding='utf-8-sig')
print(f"{len(all_reviews)} пікір сәтті сақталды!")


Обрабатываем axis-y сыворотка: https://kaspi.kz/shop/p/axis-y-syvorotka-dark-spot-correcting-glow-dlja-litsa-50-ml-104067981/?c=750000000
Найдено 54 отзывов на текущей странице для axis-y сыворотка
Найдена кнопка с XPath: //a[contains(., 'Показать ещё')]
Найдено 117 отзывов на текущей странице для axis-y сыворотка
Собрали 100 отзывов для axis-y сыворотка, переходим к следующему продукту
Собрано 100 отзывов для axis-y сыворотка

Обрабатываем велик мечты: https://kaspi.kz/shop/p/gestalt-hx1000-9027-27-5-djuim-2022-seryi-105318549/?c=750000000
Найдено 45 отзывов на текущей странице для велик мечты
Найдена кнопка с XPath: //a[contains(., 'Показать ещё')]
Найдено 108 отзывов на текущей странице для велик мечты
Собрали 100 отзывов для велик мечты, переходим к следующему продукту
Собрано 100 отзывов для велик мечты

Обрабатываем ватный диск: https://kaspi.kz/shop/p/bella-vatnye-diski-cotton-120-sht-100224746/?c=750000000
Найдено 90 отзывов на текущей странице для ватный диск
Найдена кнопка с

In [9]:
df

,ID,Атауы,Күні,Баға,Пікір
0,183d8506-68a5-440e-8049-c91b11aa7b68,axis-y сыворотка,16.10.2022,5.0,"Достоинства:Найс, обычно не оставляю отзыв (ле..."
1,99f3c943-0825-462d-a92a-7f82fb05c18b,axis-y сыворотка,16.08.2023,5.0,Мен қорқып едім подделкасын салып жіберетін шы...
2,18323017-ef37-4ade-8345-ca5acd0faf15,axis-y сыворотка,28.06.2024,1.0,"Отзывтарға қарап заказ бергем, подделкасын сал..."
3,167a85ab-f71b-44e9-b144-eb1a3cc622d0,axis-y сыворотка,27.06.2023,5.0,Достоинства:Все супер! Доставили быстро и поло...
4,b398cee2-5a6c-424e-81c9-0c9dcf053fa0,axis-y сыворотка,29.03.2023,5.0,Спасибо магазину Красотка и Каспи за предостав...
...,...,...,...,...,...
495,2f825275-d1c2-449f-a29f-09b57d4c954e,айфончик,07.11.2023,1.0,"Обслуживание ужасное, с доставкой опоздали на ..."
496,25758d69-08e5-4b3f-9773-cdad151ab72d,айфончик,31.07.2022,3.0,Достоинства:Хорошая батарейка и камера просто ...
497,6da091c6-c849-4a12-973e-3f0d84327a5b,айфончик,15.03.2022,5.0,"Достоинства:Красивый, уверенно и приятно лежит..."
498,419a88b7-1093-4300-ad00-368ee9e3f9bf,айфончик,12.03.2025,5.0,"Ұлыма сыйлыққа алғанмын, ұнады, курьерге рақме..."
